In [2]:
!pip install pymoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 18.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 32.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [pymoo]


In [3]:
import numpy as np
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.sampling.rnd import IntegerRandomSampling
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.repair.rounding import RoundingRepair
from pymoo.optimize import minimize

class KubernetesClusterProblem(ElementwiseProblem):
    def __init__(self):
        # x1: Replicas da API (2 a 40), x2: Worker Nodes (1 a 12)
        super().__init__(
            n_var=2,
            n_obj=2,
            xl=np.array([2, 1]),
            xu=np.array([40, 12]),
            vtype=int
        )

    def _evaluate(self, x, out, *args, **kwargs):
        replicas, nodes = x[0], x[1]

        # TODO 1: Calcule o Tempo de Resposta (ms): (600 / replicas) + (200 / nodes)
        f1_tempo = (600.0 / replicas) + (200.0 / nodes)

        # TODO 2: Calcule o Custo Mensal (USD): (replicas * 12) + (nodes * 85)
        f2_custo = (replicas * 12.0) + (nodes * 85.0)

        out["F"] = [f1_tempo, f2_custo]

# Instancia o problema
problem = KubernetesClusterProblem()

# TODO 3: Configure o NSGA2 com pop_size=25 utilizando IntegerRandomSampling e RoundingRepair
# no crossover SBX e na mutacao PM (consulte o Exercicio 1 como referencia).
algorithm = NSGA2(
    pop_size=25,
    sampling=IntegerRandomSampling(),
    crossover=SBX(prob=1.0, eta=15, vtype=float, repair=RoundingRepair()),
    mutation=PM(prob=1.0, eta=20, vtype=float, repair=RoundingRepair())
)

# TODO 4: Execute a minimizacao por 35 geracoes (seed=10) e imprima as solucoes X e F.
res = minimize(
    problem,
    algorithm,
    termination=('n_gen', 35),
    seed=10,
    verbose=False
)

print("=== EXERCÍCIO 2: RESULTADOS DA OTIMIZAÇÃO (KUBERNETES) ===\n")
if res.X is not None:
    print(f"Encontradas {len(res.X)} soluções na fronteira de Pareto:\n")
    print(f"{'Solução':<8} | {'Réplicas (x1)':<15} | {'Nodes (x2)':<12} | {'Tempo f1 (ms)':<15} | {'Custo f2 (USD)':<15}")
    print("-" * 75)
    for i in range(len(res.X)):
        sol_x = res.X[i]
        sol_f = res.F[i]
        print(f"{i+1:<8} | {int(round(sol_x[0])):<15} | {int(round(sol_x[1])):<12} | {sol_f[0]:<15.2f} | {sol_f[1]:<15.2f}")
else:
    print("Nenhuma solução encontrada.")

=== EXERCÍCIO 2: RESULTADOS DA OTIMIZAÇÃO (KUBERNETES) ===

Encontradas 25 soluções na fronteira de Pareto:

Solução  | Réplicas (x1)   | Nodes (x2)   | Tempo f1 (ms)   | Custo f2 (USD) 
---------------------------------------------------------------------------
1        | 2               | 1            | 500.00          | 109.00         
2        | 40              | 12           | 31.67           | 1500.00        
3        | 3               | 1            | 400.00          | 121.00         
4        | 37              | 7            | 44.79           | 1039.00        
5        | 40              | 9            | 37.22           | 1245.00        
6        | 4               | 1            | 350.00          | 133.00         
7        | 40              | 10           | 35.00           | 1330.00        
8        | 15              | 4            | 90.00           | 520.00         
9        | 15              | 2            | 140.00          | 350.00         
10       | 7               | 5     